# Setup

In [7]:
from pprint import pprint


# Seed
seed = 42

# 1. Rollout

- Trajectory: a collection of observations and actions for one task.
- Rollout: a trajectory collection for agent learning (e.g. SFT), by performing the task with the given policy.

## 1.1. Generation

In [13]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from math import isclose
import json
from uuid import uuid4
from pathlib import Path


# 1. Task
buggy_code = """
def average(values):
    return sum(values) // len(values)
"""

tests = [
    ([], 0.0),
    ([0], 0.0),
    ([1, 2], 1.5),
    ([-2, -3], -2.5),
    ([0.1, 0.2, 0.3], 0.2),
]


# 2. Chat
model_config = dict(
    model='qwen3:4b',
    temperature=0,
    top_k=5,
    top_p=0.9,
    reasoning=True,
    seed=seed,    
)
chat = ChatOllama(**model_config)


# 3. Tool
@tool
def test_code(code: str) -> str:
    """Test a complete average(values) function on each test case."""
    results = []

    for values, answer in tests:
        result = None   # returned value from average(values).
        error = None    # error msg if exists.

        try:
            namespace = {}
            exec(code, namespace)
            ftn = namespace['average']
            result = ftn(values)
            passed = isclose(answer, result)
        except Exception as exception:
            result = None
            passed = False
            error = f"{type(exception).__name__}: {exception}"

        results.append(
            dict(
                input=values,
                answer=answer,
                result=result,
                passed=passed,
                error=error,
            )
        )

    return json.dumps(
        {
            'all_passed': all(row['passed'] for row in results),
            'results': results,
        }
    )


# 4. Agent
middleware = [ModelCallLimitMiddleware(run_limit=5, exit_behavior='error')]

agent = create_agent(
    model=chat,
    tools=[test_code],
    middleware=middleware,
)


# 5. Rollout
system_prompt = (
    'You repair Python functions using test_code. '
    'First test the original code. Then revise it using the test feedback. '
    'Make one tool call at a time. Stop when all tests pass. '
    'The delivered repair is the last code submitted to test_code. '
    'Finish with a short status, not another code version. '
    'Never invent test results.'
)
user_prompt = (
    "Fix average(values). Keep its name and arguments. "
    "Return 0.0 for empty input; otherwise return the arithmetic mean.\n"
    f"Original code:\n{buggy_code}"
)

messages = [
    SystemMessage(system_prompt),
    HumanMessage(user_prompt),
]

run_error = None
printed = 0

try:
    for state in agent.stream(
        {'messages': messages},
        config={'recursion_limit': 50},
        stream_mode='values',
    ):
        # Each snapshot already contains the accumulated messages.
        messages = list(state['messages'])

        for message in messages[printed:]:
            message.pretty_print()
        printed = len(messages)

except Exception as exc:
    # Preserve the partial trajectory when execution fails.
    run_error = f'{type(exc).__name__}: {exc}'
test_messages = [
    m for m in messages
    if isinstance(m, ToolMessage) and m.name == 'test_code'
]
last_test = test_messages[-1] if test_messages else None
report = json.loads(last_test.content) if last_test else {}


# 6. Dataset
trajectory = {
    'format': 'langchain_raw_v1',
    'task_id': 'average_v1',
    'run_id': uuid4().hex,
    'model_config': model_config,
    'tests': tests,
    'tools': [{
        'type': 'function',
        'function': {
            'name': test_code.name,
            'description': test_code.description,
            'parameters': test_code.args_schema.model_json_schema(),
        },
    }],
    'messages': [m.model_dump(mode='json') for m in messages],
    'outcome': {
        'last_test_call_id': last_test.tool_call_id if last_test else None,
        'last_test_passed': report.get('all_passed', False),
        'run_error': run_error,
    },
}

log_path = Path('../tmp/logs/react_trajectories.jsonl')
log_path.parent.mkdir(parents=True, exist_ok=True)

with log_path.open('a', encoding='utf-8') as file:
    file.write(json.dumps(trajectory, ensure_ascii=False) + '\n')

print('Outcome:', trajectory['outcome'])
print('Saved:', log_path.resolve())

================================ System Message ================================

You repair Python functions using test_code. First test the original code. Then revise it using the test feedback. Make one tool call at a time. Stop when all tests pass. The delivered repair is the last code submitted to test_code. Finish with a short status, not another code version. Never invent test results.
================================ Human Message =================================

Fix average(values). Keep its name and arguments. Return 0.0 for empty input; otherwise return the arithmetic mean.
Original code:

def average(values):
    return sum(values) // len(values)

================================== Ai Message ==================================
Tool Calls:
  test_code (ba6fc170-ce70-4daa-986e-82f04b7e434d)
 Call ID: ba6fc170-ce70-4daa-986e-82f04b7e434d
  Args:
    code: def average(values):
    return sum(values) // len(values)
================================= Tool Message ===============

## 1.2. Data Preparation

In [ ]:
from pathlib import Path
import json


# 1. Convert langchain msg -> training format.
def convert_message(message: dict) -> dict:
    roles = {
        'system': 'system',
        'human': 'user',
        'ai': 'assistant',
        'tool': 'tool',
    }
    result = {
        'role': roles[message['type']],
        'content': message['content'],
    }

    if message['type'] == 'ai' and message.get('tool_calls'):
        result['tool_calls'] = [
            {
                'id': call['id'],
                'type': 'function',
                'function': {
                    'name': call['name'],
                    'arguments': call['args'],
                },
            }
            for call in message['tool_calls']
        ]

    elif message['type'] == 'tool':
        result.update(
            name=message['name'],
            tool_call_id=message['tool_call_id'],
        )

    return result


# 2. Select trajectories for training.
source = Path('../tmp/logs/react_trajectories.jsonl')
rows = []

for line in source.read_text(encoding='utf-8').splitlines():
    run = json.loads(line)
    pprint(run)

    outcome = run['outcome']            # run results.
    run_error = outcome['run_error']    # run error.
    last_test_passed = outcome['last_test_passed']      # whether the last test is passed.
    raw = run['messages']

    if any([
        (run_error or not last_test_passed),                # if there is run error or test failed,
        (not raw or raw[-1]['type'] != 'ai'),               # if messages is empty or not ai type,
        (raw[-1].get('tool_calls')),                        # if no tool calls,
        (any(m.get('invalid_tool_calls') for m in raw)),    # if any invalid tool calls,
    ]):
        continue

    call_ids = [
        call['id']
        for message in raw
        for call in message.get('tool_calls', [])
    ]
    result_ids = [
        message['tool_call_id']
        for message in raw
        if message['type'] == 'tool'
    ]

    if not call_ids or any(not call_id for call_id in call_ids):
        continue
    if len(set(call_ids)) != len(call_ids) or sorted(call_ids) != sorted(result_ids):
        continue

    rows.append({
        'messages': [convert_message(message) for message in raw],
        'tools': run['tools'],
    })

if not rows:
    raise ValueError(
        'No complete successful trajectories. Collect or inspect rollouts first.'
    )


# 3. Save
output = Path('../tmp/logs/agent_sft.jsonl')
output.write_text(
    '\n'.join(json.dumps(row, ensure_ascii=False) for row in rows) + '\n',
    encoding='utf-8',
)

print('Training candidates:', len(rows))
print(json.dumps(rows[0], indent=2, ensure_ascii=False))

{'format': 'langchain_raw_v1',
 'messages': [{'additional_kwargs': {},
               'content': 'You repair Python functions using test_code. First '
                          'test the original code. Then revise it using the '
                          'test feedback. Make one tool call at a time. Stop '
                          'when all tests pass. The delivered repair is the '
                          'last code submitted to test_code. Finish with a '
                          'short status, not another code version. Never '
                          'invent test results.',
               'id': '24b86699-2f2d-43c0-98db-62af6eecfc6b',
               'name': None,
               'response_metadata': {},
               'type': 'system'},
              {'additional_kwargs': {},
               'content': 'Fix average(values). Keep its name and arguments. '
                          'Return 0.0 for empty input; otherwise return the '
                          'arithmetic mean.\n'
    

# 2. Agent SFT

In [16]:
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer


# 1. Load
rows = [
    json.loads(line)
    for line in Path('../tmp/logs/agent_sft.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]

dataset = Dataset.from_list(rows)
model_name = 'Qwen/Qwen3-4B'
output_dir = '../tmp/logs/agent_sft_lora'
use_bf16 = torch.cuda.is_bf16_supported()

tokenizer = AutoTokenizer.from_pretrained(model_name)


# 2. Configure assistant-only SFT with LoRA.
trainer = SFTTrainer(
    model=model_name,
    processing_class=tokenizer,
    train_dataset=dataset,
    peft_config=LoraConfig(
        task_type='CAUSAL_LM',
        r=8,
        lora_alpha=16,
        target_modules='all-linear',
    ),
    args=SFTConfig(
        output_dir=output_dir,
        assistant_only_loss=True,
        max_length=None,                 # Do not silently truncate trajectories.
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=10,                    # Pipeline check, not a full experiment.
        learning_rate=1e-4,
        logging_steps=1,
        save_strategy='no',
        report_to='none',
        bf16=use_bf16,
        fp16=False,
        model_init_kwargs={
            'dtype': torch.bfloat16 if use_bf16 else torch.float32,
        },
        seed=42,
    ),
)


# 3. Inspect the actual training targets.
if max(len(row['input_ids']) for row in trainer.train_dataset) > 4096:
    raise ValueError(
        'A trajectory exceeds the demo token budget. Review its length first.'
    )

batch = trainer.data_collator([trainer.train_dataset[0]])
labels = batch['labels'][0]
target_ids = labels[labels != -100].tolist()

if not target_ids:
    raise ValueError('No training targets. Check the assistant mask and chat template.')

print('Target preview:', tokenizer.decode(target_ids)[:500])


# 4. Train and save the adapter.
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Target preview: <think>

</think>

<tool_call>
{"name": "test_code", "arguments": {"code": "def average(values):\n    return sum(values) // len(values)"}}
</tool_call><|im_end|>
<think>

</think>

<tool_call>
{"name": "test_code", "arguments": {"code": "def average(values):\n    if len(values) == 0:\n        return 0.0\n    return sum(values) / len(values)"}}
</tool_call><|im_end|>
<think>

</think>

All tests passed. Final code: def average(values): if len(values) == 0: return 0.0 else: return sum(values) / le


Step,Training Loss
1,1.144041
2,1.012693
3,0.778067
4,0.585795
5,0.390940
6,0.292052
7,0.229578
8,0.188905
9,0.160460
10,0.143260


('../tmp/logs/agent_sft_lora\\tokenizer_config.json',
 '../tmp/logs/agent_sft_lora\\chat_template.jinja',
 '../tmp/logs/agent_sft_lora\\tokenizer.json')

# 3. Reward / Credit Assignments

- Reinforcement learning: the agent learns to choose more actions that rewards better.
  - Credit assignment: how to connect the result and the action. how much each action did contribute to the result?
  - Reward: how to score for the action.
    - Advantage: how much did it do better than a baseline?
    - Baseline
      - PPO: a prediction of the future return expected from the current status, predicted by the value model or critic.
      - GRPO: an average of the results of multiple trials.
  - PPO / GRPO: how to update the policy.
  - Policy: how to choose the action based on observations -> this is what is trained!
- Policy gradient
  - log-probability of chosen action x performance signal -> update policy

## 3.1. PPO

- Proximal Policy Optimization

> Note) `trl.experimental.ppo` is removed in the recent version. Use `trl==1.8.0` or `OpenRLHF` or `verl`.

In [ ]:
from math import isclose
from types import SimpleNamespace

import torch
from torch import nn
from datasets import Dataset
from peft import LoraConfig
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    PreTrainedModel, PretrainedConfig, set_seed,
)
from trl.experimental.ppo import PPOConfig, PPOTrainer


# Task: select one implementation, then score it with actual tests.
patches = {
    'A': lambda v: sum(v) // len(v) if v else 0.0,
    'B': lambda v: sum(v) / len(v) if v else 0.0,
    'C': lambda v: 0.0,
}
tests = [([], 0.0), ([1, 2], 1.5), ([-2, -3], -2.5)]


def score_patch(text: str) -> float:
    fn = patches.get(text.strip())
    if fn is None:
        return 0.0
    return float(all(isclose(fn(x), y, abs_tol=1e-9) for x, y in tests))


request = (
    'Choose a correct implementation of average(v). Return only A, B, or C.\n'
    'A: sum(v) // len(v) if v else 0.0\n'
    'B: sum(v) / len(v) if v else 0.0\n'
    'C: 0.0\n'
    'An empty input must return 0.0; otherwise return the arithmetic mean.'
)

set_seed(42)
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled PyTorch installation is required.')

model_name = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': request}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)

# The identical prompt keeps the completion boundary fixed in this tiny demo.
prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
prompt_length = len(prompt_ids)
dataset = Dataset.from_dict({'input_ids': [prompt_ids] * 8})


# Adapter: expose our deterministic test score through PPO's reward-model API.
# This is NOT a learned reward model or a critic.
class TestBackbone(nn.Module):
    def forward(self, input_ids, attention_mask, **kwargs):
        texts = [
            tokenizer.decode(
                row[prompt_length:][mask[prompt_length:].bool()].tolist(),
                skip_special_tokens=True,
            )
            for row, mask in zip(input_ids, attention_mask)
        ]
        scores = torch.tensor(
            [score_patch(text) for text in texts],
            dtype=torch.float32, device=input_ids.device,
        )
        # PPO selects the final non-padding position for the terminal score.
        hidden = scores[:, None, None].expand(-1, input_ids.shape[1], 1)
        return SimpleNamespace(hidden_states=(hidden,))


class TestReward(PreTrainedModel):
    config_class = PretrainedConfig
    base_model_prefix = 'backbone'

    def __init__(self):
        super().__init__(PretrainedConfig())
        self.backbone = TestBackbone()
        self.score = nn.Identity()


# Policy: train a LoRA adapter. Critic: train only its scalar value head.
policy = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
critic = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=1, dtype=torch.float32,
    pad_token_id=tokenizer.pad_token_id,
)
critic.base_model.requires_grad_(False)

trainer = PPOTrainer(
    model=policy,
    ref_model=None,  # With PEFT, the frozen base model supplies the reference.
    reward_model=TestReward(),
    value_model=critic,
    processing_class=tokenizer,
    train_dataset=dataset,
    eval_dataset=dataset,  # API requirement only; evaluation is disabled.
    peft_config=LoraConfig(
        task_type='CAUSAL_LM', r=8, lora_alpha=16,
        target_modules=['q_proj', 'v_proj'],
    ),
    args=PPOConfig(
        output_dir='ppo_patch',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        total_episodes=8,
        num_mini_batches=1,
        num_ppo_epochs=2,
        response_length=4,
        stop_token='eos',
        local_rollout_forward_batch_size=1,
        num_sample_generations=0,
        learning_rate=1e-4,
        gamma=1.0, lam=0.95, cliprange=0.2, kl_coef=0.02,
        bf16=False, fp16=False, gradient_checkpointing=False,
        optim='adamw_torch', report_to='none',
        logging_steps=1, save_strategy='no', seed=42,
    ),
)
trainer.train()
trainer.save_model('../tmp/ppo_patch')
tokenizer.save_pretrained('../tmp/ppo_patch')


## 3.2. GRPO

- Group Relative Policy Optimization

In [6]:
from math import isclose

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, set_seed
from trl import GRPOConfig, GRPOTrainer


# Task: select one implementation, then score it with actual tests.
patches = {
    'A': lambda v: sum(v) // len(v) if v else 0.0,
    'B': lambda v: sum(v) / len(v) if v else 0.0,
    'C': lambda v: 0.0,
}
tests = [([], 0.0), ([1, 2], 1.5), ([-2, -3], -2.5)]


def score_patch(text: str) -> float:
    fn = patches.get(text.strip())
    if fn is None:
        return 0.0
    return float(all(isclose(fn(x), y, abs_tol=1e-9) for x, y in tests))


request = (
    'Choose a correct implementation of average(v). Return only A, B, or C.\n'
    'A: sum(v) // len(v) if v else 0.0\n'
    'B: sum(v) / len(v) if v else 0.0\n'
    'C: 0.0\n'
    'An empty input must return 0.0; otherwise return the arithmetic mean.'
)

set_seed(42)
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled PyTorch installation is required.')

model_name = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': request}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)

# Raw strings produce string completions in the reward callback.
dataset = Dataset.from_dict({'prompt': [prompt] * 8})


def reward_func(completions, **kwargs):
    return [score_patch(text) for text in completions]


trainer = GRPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    train_dataset=dataset,
    reward_funcs=reward_func,
    peft_config=LoraConfig(
        task_type='CAUSAL_LM', r=8, lora_alpha=16,
        target_modules=['q_proj', 'v_proj'],
    ),
    args=GRPOConfig(
        output_dir='../tmp/grpo_patch',
        model_init_kwargs={'dtype': torch.float32},
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=4,
        max_steps=4,
        max_completion_length=4,
        temperature=1.0,
        num_iterations=2,
        loss_type='grpo', scale_rewards='group',
        beta=0.02,
        learning_rate=1e-4,
        bf16=False, fp16=False, gradient_checkpointing=False,
        use_vllm=False, optim='adamw_torch', report_to='none',
        logging_steps=1, save_strategy='no', seed=42,
    ),
)
trainer.train()
trainer.save_model('../tmp/grpo_patch')
tokenizer.save_pretrained('../tmp/grpo_patch')


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000


('../tmp/grpo_patch\\tokenizer_config.json',
 '../tmp/grpo_patch\\chat_template.jinja',
 '../tmp/grpo_patch\\tokenizer.json')

## 3.3. DPO

- Direct Preference Optimization

In [7]:
from math import isclose

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer


# Task: select one implementation, then score it with actual tests.
patches = {
    'A': lambda v: sum(v) // len(v) if v else 0.0,
    'B': lambda v: sum(v) / len(v) if v else 0.0,
    'C': lambda v: 0.0,
}
tests = [([], 0.0), ([1, 2], 1.5), ([-2, -3], -2.5)]


def score_patch(text: str) -> float:
    fn = patches.get(text.strip())
    if fn is None:
        return 0.0
    return float(all(isclose(fn(x), y, abs_tol=1e-9) for x, y in tests))


request = (
    'Choose a correct implementation of average(v). Return only A, B, or C.\n'
    'A: sum(v) // len(v) if v else 0.0\n'
    'B: sum(v) / len(v) if v else 0.0\n'
    'C: 0.0\n'
    'An empty input must return 0.0; otherwise return the arithmetic mean.'
)

set_seed(42)
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA-enabled PyTorch installation is required.')

model_name = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': request}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)

# Build preferences from test outcomes, not from the model's claim.
chosen = [action for action in patches if score_patch(action) == 1.0]
rejected = [action for action in patches if score_patch(action) == 0.0]
dataset = Dataset.from_list([
    {'prompt': prompt, 'chosen': good, 'rejected': bad}
    for good in chosen for bad in rejected
] * 4)


trainer = DPOTrainer(
    model=model_name,
    processing_class=tokenizer,
    train_dataset=dataset,
    peft_config=LoraConfig(
        task_type='CAUSAL_LM', r=8, lora_alpha=16,
        target_modules=['q_proj', 'v_proj'],
    ),
    args=DPOConfig(
        output_dir='../tmp/dpo_patch',
        model_init_kwargs={'dtype': torch.float32},
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        max_steps=4,
        max_length=512,
        loss_type='sigmoid', beta=0.1,
        learning_rate=1e-4,
        bf16=False, fp16=False, gradient_checkpointing=False,
        optim='adamw_torch', report_to='none',
        logging_steps=1, save_strategy='no', seed=42,
    ),
)
trainer.train()
trainer.save_model('../tmp/dpo_patch')
tokenizer.save_pretrained('../tmp/dpo_patch')


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Step,Training Loss
1,0.693147
2,0.898134
3,0.354912
4,0.530032


('../tmp/dpo_patch\\tokenizer_config.json',
 '../tmp/dpo_patch\\chat_template.jinja',
 '../tmp/dpo_patch\\tokenizer.json')